
Databricks free : https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2332341503038484?o=7474657872577658

https://customer-academy.databricks.com/learn/courses/3959/stream-processing-and-analysis-with-apache-spark/lessons/74443:4278/stream-aggregations-windows-and-watermarks?hash=3e063f193bec2e8ca7cdf56ffbdced1657bcc087&generated_by=409740

https://community.databricks.com/t5/learning-events/advanced-learning-festival-15-june-06-july-2026/ev-p/156932

## Lecture - Stream Aggregations, Windows and Watermarks

Sliding Window Example

Get the number of credit card authorizations for the last 60 seconds, every 30 seconds:

In [0]:

#from pyspark.sql.functions import window, count

#authorizations = ...  # Schema: { timestamp: Timestamp, card_id: String, amount: Double }

#windowed_auths = (
#    authorizations
#    .groupBy(
#        window(
#            timeColumn=authorizations.timestamp,   # the event-time column
#            windowDuration="60 seconds",           # each window covers 60 seconds
#            slideDuration="30 seconds"             # a new window starts every 30 seconds
#        ),
#        authorizations.card_id                     # additional grouping key
#    )
#    .agg(count("*").alias("transaction_count"))
#)

# Start a streaming query to send the output to a sink
#Omit slideDuration and the same call produces tumbling windows. The result includes a window struct column with start and end fields.

O que faz o exemplo de Sliding Window
Esse trecho usa window(...) em conjunto com groupBy(...) e agg(...) no PySpark para contar quantas autorizações de cartão de crédito (transaction_count) ocorreram por card_id em janelas temporais deslizantes baseadas no tempo do evento (timestamp) — e não no tempo de processamento do Spark.

windowDuration="60 seconds": cada janela tem duração de 60 segundos.

slideDuration="30 seconds": uma nova janela começa a cada 30 segundos.

Sobreposição de eventos: como slideDuration (30s) é menor que windowDuration (60s), as janelas se sobrepõem. Uma autorização feita às 10:00:40 cai tanto na janela [10:00:00, 10:01:00) quanto na janela [10:00:30, 10:01:30).

Coluna window (struct): o resultado gera uma coluna window contendo os campos start e end (início e fim de cada intervalo).

Diferença para Tumbling Window: se você omitir slideDuration, o Spark assume o mesmo valor de windowDuration (60s), criando janelas contíguas e sem sobreposição (tumbling windows).

### Exemplo 1: Como as janelas se sobrepõem (com dados de exemplo em batch/memory para visualizar)

Este exemplo simula eventos de um mesmo cartão (CARD-1) em horários próximos para mostrar na prática como uma mesma transação entra em mais de uma janela:

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, to_timestamp, window

spark = SparkSession.builder.getOrCreate()

# Dados de exemplo com autorizações em segundos diferentes
events_df = (
    spark.createDataFrame(
        [
            ("2026-11-26 10:00:10", "CARD-1", 50.0),
            ("2026-11-26 10:00:40", "CARD-1", 120.0),  # Entra em [10:00:00-10:01:00) e [10:00:30-10:01:30)
            ("2026-11-26 10:01:05", "CARD-1", 80.0),   # Entra em [10:00:30-10:01:30) e [10:01:00-10:02:00)
        ],
        ["ts_str", "card_id", "amount"],
    )
    .withColumn("timestamp", to_timestamp(col("ts_str")))
    .drop("ts_str")
)
events_df.show()

+-------+------+-------------------+
|card_id|amount|          timestamp|
+-------+------+-------------------+
| CARD-1|  50.0|2026-11-26 10:00:10|
| CARD-1| 120.0|2026-11-26 10:00:40|
| CARD-1|  80.0|2026-11-26 10:01:05|
+-------+------+-------------------+



In [0]:

# Sliding window de 60s avançando a cada 30s por card_id
sliding_df = (
    events_df.groupBy(
        window(
            timeColumn=col("timestamp"),
            windowDuration="60 seconds",
            slideDuration="30 seconds",
        ),
        col("card_id"),
    )
    .agg(count("*").alias("transaction_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("card_id"),
        col("transaction_count"),
    )
    .orderBy("window_start", "card_id")
)

sliding_df.show(truncate=False)

+-------------------+-------------------+-------+-----------------+
|window_start       |window_end         |card_id|transaction_count|
+-------------------+-------------------+-------+-----------------+
|2026-11-26 09:59:30|2026-11-26 10:00:30|CARD-1 |1                |
|2026-11-26 10:00:00|2026-11-26 10:01:00|CARD-1 |2                |
|2026-11-26 10:00:30|2026-11-26 10:01:30|CARD-1 |2                |
|2026-11-26 10:01:00|2026-11-26 10:02:00|CARD-1 |1                |
+-------------------+-------------------+-------+-----------------+



In [0]:
# página 18

## Exemplo 2: Streaming completo com readStream, watermark, agregação e writeStream

Em agregações por janela no Structured Streaming, costuma-se usar with Watermark(...) para controlar até que ponto eventos atrasados (late data) são aceitos e para permitir que o Spark limpe o estado antigo. Como a contagem muda à medida que novos eventos chegam na mesma janela, o outputMode("update") é muito comum para janelas abertas:

In [0]:
import json
import os
from pyspark.sql.functions import col, count, window
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

base_dir = "./sliding_window_demo"
input_dir = f"{base_dir}/authorizations_input"
checkpoint_dir = f"{base_dir}/checkpoint_sliding"

os.makedirs(input_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

# Gera um arquivo JSON inicial de autorizações
sample_auths = [
    {"timestamp": "2026-11-26T10:00:10.000Z", "card_id": "CARD-1", "amount": 45.0},
    {"timestamp": "2026-11-26T10:00:35.000Z", "card_id": "CARD-1", "amount": 90.0},
    {"timestamp": "2026-11-26T10:00:50.000Z", "card_id": "CARD-2", "amount": 30.0},
]

In [0]:


with open(f"{input_dir}/auth_01.json", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(row) for row in sample_auths) + "\n")

In [0]:

# Schema explícito da fonte de streaming
auth_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("card_id", StringType(), True),
    StructField("amount", DoubleType(), True),
])

# Fontes de arquivo no Spark exigem caminho absoluto
input_path = "file:/Workspace/Users/fabieneaulas@gmail.com/APACHE_SPARK/StreamProcessingandAnalysis/sliding_window_demo/authorizations_input"

authorizations_stream = (
    spark.readStream
    .format("json")
    .schema(auth_schema)
    .load(input_path)
)

windowed_auths = (
    authorizations_stream
    .withWatermark("timestamp", "2 minutes")
    .groupBy(
        window(
            timeColumn=col("timestamp"),
            windowDuration="60 seconds",
            slideDuration="30 seconds",
        ),
        col("card_id"),
    )
    .agg(count("*").alias("transaction_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("card_id"),
        col("transaction_count"),
    )
)

# checkpointLocation exigido para queries de streaming neste workspace
checkpoint_path = "/Volumes/main/default/streaming_demo/checkpoint_sliding"

display(windowed_auths, checkpointLocation=checkpoint_path)

Checkpointing to /Volumes/main/default/streaming_demo/checkpoint_sliding


In [0]:
# Inicia a streaming query para memória com availableNow=True para teste rápido
auth_query = (
    windowed_auths.writeStream
    .format("memory")
    .queryName("auth_sliding_window_view")
    .outputMode("update")
    .option("checkpointLocation", "/Volumes/main/default/streaming_demo/checkpoint_sliding_memory")
    .trigger(availableNow=True)
    .start()
)

In [0]:

auth_query.awaitTermination()

spark.sql("""
    SELECT window_start, window_end, card_id, transaction_count
    FROM auth_sliding_window_view
    ORDER BY window_start, card_id
""").show(truncate=False)

+-------------------+-------------------+-------+-----------------+
|window_start       |window_end         |card_id|transaction_count|
+-------------------+-------------------+-------+-----------------+
|2026-11-26 09:59:30|2026-11-26 10:00:30|CARD-1 |1                |
|2026-11-26 10:00:00|2026-11-26 10:01:00|CARD-1 |2                |
|2026-11-26 10:00:00|2026-11-26 10:01:00|CARD-2 |1                |
|2026-11-26 10:00:30|2026-11-26 10:01:30|CARD-1 |1                |
|2026-11-26 10:00:30|2026-11-26 10:01:30|CARD-2 |1                |
+-------------------+-------------------+-------+-----------------+



## Exemplo 3: Comparação direta com Tumbling Window (sem slideDuration)

Se você quiser ver a diferença no mesmo conjunto de dados sem sobreposição entre janelas de 60 segundos:

In [0]:
tumbling_auths = (
    events_df.groupBy(
        window(
            timeColumn=col("timestamp"),
            windowDuration="60 seconds",  # Sem slideDuration -> janelas de 60s sem sobreposição
        ),
        col("card_id"),
    )
    .agg(count("*").alias("transaction_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("card_id"),
        col("transaction_count"),
    )
    .orderBy("window_start", "card_id")
)

tumbling_auths.show(truncate=False)

+-------------------+-------------------+-------+-----------------+
|window_start       |window_end         |card_id|transaction_count|
+-------------------+-------------------+-------+-----------------+
|2026-11-26 10:00:00|2026-11-26 10:01:00|CARD-1 |2                |
|2026-11-26 10:01:00|2026-11-26 10:02:00|CARD-1 |1                |
+-------------------+-------------------+-------+-----------------+



##### Handling Late Arriving Data explica como o withWatermark controla eventos atrasados em agregações por janela no Structured Streaming (como aceitar eventos até ao limiar definido, descartar eventos demasiado atrasados, limpar estado antigo e permitir o modo append quando o resultado da janela já se considera definitivo).  


 Seguem dois exemplos práticos com ficheiros JSON locais (sem depender de volumes do Unity Catalog) para veres na prática como funciona o watermark com janelas deslizantes.   

## Exemplo 1: Evento atrasado dentro do watermark vs. evento demasiado atrasado com outputMode("append")


Neste cenário, usamos um watermark de 2 minutos ("2 minutes"):

- Se o maior timestamp visto até agora for 10:05:00, o watermark fica em 10:03:00 (10:05:00 - 2 min).

- Um evento com timestamp em 10:03:30 ainda vai dentro do limiar e pode ser incluído.

- Um evento com timestamp em 10:01:00 já ficou para trás do watermark e pode ser descartado.  

- Com outputMode("append"), o Spark só emite o resultado de uma janela quando o watermark avança para além do fim dessa janela.   

In [0]:
import json
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, window
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

# Prepara diretórios locais limpos para o teste
base_dir = "./watermark_demo_append"
shutil.rmtree(base_dir, ignore_errors=True)
input_dir = f"{base_dir}/events"
checkpoint_dir = f"{base_dir}/checkpoint"
os.makedirs(input_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

spark = SparkSession.builder.getOrCreate()

auth_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("card_id", StringType(), True),
    StructField("amount", DoubleType(), True),
])

# 1.º lote: eventos iniciais na janela [10:00:00, 10:01:00)
batch_1 = [
    {"timestamp": "2026-11-26T10:00:10.000Z", "card_id": "CARD-1", "amount": 25.0},
    {"timestamp": "2026-11-26T10:00:40.000Z", "card_id": "CARD-1", "amount": 50.0},
]
with open(f"{input_dir}/b1.json", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(r) for r in batch_1) + "\n")

In [0]:
# 2.º lote: avança o tempo para 10:03:20 (janela [10:00:00, 10:01:00) ainda não passou do watermark de 2 min)
# e inclui um evento atrasado mas válido às 10:00:50
batch_2 = [
    {"timestamp": "2026-11-26T10:00:50.000Z", "card_id": "CARD-1", "amount": 15.0},
    {"timestamp": "2026-11-26T10:03:20.000Z", "card_id": "CARD-1", "amount": 70.0},
]
with open(f"{input_dir}/b2.json", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(r) for r in batch_2) + "\n")

# 3.º lote: avança o tempo para 10:05:00 -> watermark passa a 10:03:00,
# finalizando janelas antigas em append e descartando eventos anteriores a 10:03:00
batch_3 = [
    {"timestamp": "2026-11-26T10:05:00.000Z", "card_id": "CARD-1", "amount": 100.0},
    {"timestamp": "2026-11-26T10:01:00.000Z", "card_id": "CARD-1", "amount": 999.0},  # Muito atrasado
]
with open(f"{input_dir}/b3.json", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(r) for r in batch_3) + "\n")

In [0]:
authorizations = (
    spark.readStream
    .format("json")
    .schema(auth_schema)
    .option("maxFilesPerTrigger", 1)  # Processa 1 ficheiro por micro-batch para observar o avanço
    .load("file:/Workspace/Users/fabieneaulas@gmail.com/APACHE_SPARK/StreamProcessingandAnalysis/watermark_demo_append/events")
)

windowed_auths = (
    authorizations
    .withWatermark("timestamp", "2 minutes")  # Aceita eventos até 2 minutos de atraso
    .groupBy(
        window(col("timestamp"), "60 seconds", "30 seconds"),
        col("card_id"),
    )
    .agg(count("*").alias("transaction_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("card_id"),
        col("transaction_count"),
    )
)

In [0]:



# Com outputMode("append"), só saem resultados das janelas já encerradas pelo watermark[cite: 13]
query = (
    windowed_auths.writeStream
    .format("memory")
    .queryName("finalized_windows_view")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/main/default/streaming_demo/checkpoint_watermark_append")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

spark.sql("""
    SELECT window_start, window_end, card_id, transaction_count
    FROM finalized_windows_view
    ORDER BY window_start, window_end, card_id
""").show(truncate=False)

+-------------------+-------------------+-------+-----------------+
|window_start       |window_end         |card_id|transaction_count|
+-------------------+-------------------+-------+-----------------+
|2026-11-26 09:59:30|2026-11-26 10:00:30|CARD-1 |1                |
|2026-11-26 10:00:00|2026-11-26 10:01:00|CARD-1 |3                |
|2026-11-26 10:00:30|2026-11-26 10:01:30|CARD-1 |3                |
|2026-11-26 10:01:00|2026-11-26 10:02:00|CARD-1 |1                |
+-------------------+-------------------+-------+-----------------+



In [0]:
# página 18

## Exemplo 2: Ver atualizações intermédias com outputMode("update")

Se quiseres acompanhar as contagens a crescerem antes de a janela ser fechada definitivamente (por exemplo, durante testes ou dashboards em tempo real), podes usar outputMode("update") na mesma estrutura com withWatermark:

In [0]:
checkpoint_update_dir = "/Volumes/main/default/streaming_demo/checkpoint_update"
os.makedirs(checkpoint_update_dir, exist_ok=True)

query_update = (
    windowed_auths.writeStream
    .format("memory")
    .queryName("updating_windows_view")
    .outputMode("update")
    .option("checkpointLocation", checkpoint_update_dir)
    .trigger(availableNow=True)
    .start()
)

query_update.awaitTermination()

spark.sql("""
    SELECT window_start, window_end, card_id, transaction_count
    FROM updating_windows_view
    ORDER BY window_start, window_end, card_id
""").show(truncate=False)

+-------------------+-------------------+-------+-----------------+
|window_start       |window_end         |card_id|transaction_count|
+-------------------+-------------------+-------+-----------------+
|2026-11-26 09:59:30|2026-11-26 10:00:30|CARD-1 |1                |
|2026-11-26 10:00:00|2026-11-26 10:01:00|CARD-1 |3                |
|2026-11-26 10:00:00|2026-11-26 10:01:00|CARD-1 |2                |
|2026-11-26 10:00:30|2026-11-26 10:01:30|CARD-1 |1                |
|2026-11-26 10:00:30|2026-11-26 10:01:30|CARD-1 |3                |
|2026-11-26 10:00:30|2026-11-26 10:01:30|CARD-1 |2                |
|2026-11-26 10:01:00|2026-11-26 10:02:00|CARD-1 |1                |
|2026-11-26 10:02:30|2026-11-26 10:03:30|CARD-1 |1                |
|2026-11-26 10:03:00|2026-11-26 10:04:00|CARD-1 |1                |
|2026-11-26 10:04:30|2026-11-26 10:05:30|CARD-1 |1                |
|2026-11-26 10:05:00|2026-11-26 10:06:00|CARD-1 |1                |
+-------------------+-------------------+-------

#### Streaming Joins, tens dois padrões principais mostrados no código: Stream-static join (enriquecer um stream com dados de referência estáticos, sem gestão de estado entre micro-batches) e Stream-stream join (cruzar dois streams, exigindo watermarks em ambos os lados e um intervalo temporal para limitar o estado em memória).

Seguem dois exemplos executáveis em ficheiros JSON locais (sem dbutils nem volumes do Unity Catalog) que seguem exatamente esse processo.

### Exemplo 1: Stream-Static Join (enriquecer orders_stream com dados de clientes)

Aqui, customers_static_df é um DataFrame estático com dados de referência dos clientes, e fazemos o join diretamente sobre o streaming DataFrame orders_stream por customer_id:

In [0]:
import json
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType

base_dir = "./streaming_joins_demo"
shutil.rmtree(base_dir, ignore_errors=True)

orders_dir = f"{base_dir}/orders_stream"
payments_dir = f"{base_dir}/payments_stream"
checkpoint_static = f"{base_dir}/checkpoints/stream_static"

os.makedirs(orders_dir, exist_ok=True)
os.makedirs(payments_dir, exist_ok=True)
os.makedirs(checkpoint_static, exist_ok=True)

spark = SparkSession.builder.getOrCreate()

# Dados estáticos de referência de clientes
customers_static_df = spark.createDataFrame(
    [
        (5001, "Ana Silva", "Gold"),
        (5002, "Bruno Costa", "Silver"),
        (5003, "Carla Mendes", "Platinum"),
    ],
    ["customer_id", "customer_name", "tier"],
)

# Ficheiros JSON de pedidos (orders_stream)
orders_batch = [
    {"order_id": 1001, "customer_id": 5001, "order_time": "2026-11-26T10:00:00.000Z", "total_price": 45.0},
    {"order_id": 1002, "customer_id": 5002, "order_time": "2026-11-26T10:02:00.000Z", "total_price": 90.0},
]
with open(f"{orders_dir}/orders_01.json", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(row) for row in orders_batch) + "\n")


In [0]:

orders_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("order_time", TimestampType(), True),
    StructField("total_price", StringType(), True),
])

orders_stream = (
    spark.readStream
    .format("json")
    .schema(orders_schema)
    .load("file:/Workspace/Users/fabieneaulas@gmail.com/APACHE_SPARK/StreamProcessingandAnalysis/streaming_joins_demo/orders_stream")
)

# Stream-static join: enriquece o stream sem necessidade de estado temporal
enriched_orders = orders_stream.join(customers_static_df, "customer_id", "left")

query_static = (
    enriched_orders.writeStream
    .format("memory")
    .queryName("enriched_orders_view")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/main/default/streaming_demo/checkpoint_stream_static")
    .trigger(availableNow=True)
    .start()
)

query_static.awaitTermination()

spark.sql("""
    SELECT order_id, customer_id, customer_name, tier, order_time, total_price
    FROM enriched_orders_view
    ORDER BY order_id
""").show(truncate=False)

+--------+-----------+-------------+------+-------------------+-----------+
|order_id|customer_id|customer_name|tier  |order_time         |total_price|
+--------+-----------+-------------+------+-------------------+-----------+
|1001    |5001       |Ana Silva    |Gold  |2026-11-26 10:00:00|45.0       |
|1002    |5002       |Bruno Costa  |Silver|2026-11-26 10:02:00|90.0       |
+--------+-----------+-------------+------+-------------------+-----------+



### Exemplo 2: Stream-Stream Join com Watermarks e intervalo temporal

Neste exemplo, cruzamos orders_stream com payments_stream usando order_id e restringindo o pagamento a ocorrer entre order_time e order_time + INTERVAL 10 MINUTES, com withWatermark de 10 minutes em ambos os streams para limitar o estado:

In [0]:
from pyspark.sql.functions import expr
from pyspark.sql.types import DoubleType

checkpoint_stream_join = f"{base_dir}/checkpoints/stream_stream"
os.makedirs(checkpoint_stream_join, exist_ok=True)

# Ficheiros JSON de pagamentos (payments_stream)
payments_batch = [
    {"payment_id": 9001, "order_id": 1001, "payment_time": "2026-11-26T10:03:00.000Z", "amount_paid": 45.0},
    # Pagamento fora do intervalo de 10 minutos após o pedido 1002 (10:02 + 10m = 10:12)
    {"payment_id": 9002, "order_id": 1002, "payment_time": "2026-11-26T10:20:00.000Z", "amount_paid": 90.0},
]
with open(f"{payments_dir}/payments_01.json", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(row) for row in payments_batch) + "\n")


In [0]:

payments_schema = StructType([
    StructField("payment_id", LongType(), True),
    StructField("order_id", LongType(), True),
    StructField("payment_time", TimestampType(), True),
    StructField("amount_paid", DoubleType(), True),
])

payments_stream = (
    spark.readStream
    .format("json")
    .schema(payments_schema)
    .load("file:/Workspace/Users/fabieneaulas@gmail.com/APACHE_SPARK/StreamProcessingandAnalysis/streaming_joins_demo/payments_stream")
)

# Stream-stream join com watermark nos dois lados e janela de correspondência limitada
joined_stream = (
    orders_stream.alias("orders")
    .withWatermark("order_time", "10 minutes")
    .join(
        payments_stream.alias("payments").withWatermark("payment_time", "10 minutes"),
        expr("""
            orders.order_id = payments.order_id AND
            payment_time BETWEEN order_time AND order_time + INTERVAL 10 MINUTES
        """),
        "inner",
    )
    .select(
        col("orders.order_id"),
        col("orders.customer_id"),
        col("orders.order_time"),
        col("payments.payment_id"),
        col("payments.payment_time"),
        col("payments.amount_paid"),
    )
)

query_stream_join = (
    joined_stream.writeStream
    .format("memory")
    .queryName("matched_orders_payments_view")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/main/default/streaming_demo/checkpoint_stream_stream")
    .trigger(availableNow=True)
    .start()
)

query_stream_join.awaitTermination()

spark.sql("""
    SELECT *
    FROM matched_orders_payments_view
    ORDER BY order_id
""").show(truncate=False)

+--------+-----------+-------------------+----------+-------------------+-----------+
|order_id|customer_id|order_time         |payment_id|payment_time       |amount_paid|
+--------+-----------+-------------------+----------+-------------------+-----------+
|1001    |5001       |2026-11-26 10:00:00|9001      |2026-11-26 10:03:00|45.0       |
+--------+-----------+-------------------+----------+-------------------+-----------+

